# Getting Started with MediSafeAI

This notebook demonstrates the core features of MediSafeAI:
- Generating synthetic patient data
- Creating vital signs
- Applying differential privacy
- Simulating disease progression

## Setup

First, let's import the necessary libraries and modules.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_generator.patient_generator import PatientGenerator
from src.data_generator.vitals_generator import VitalsGenerator
from src.data_generator.disease_progression import DiseaseProgressionModel
from src.data_generator.treatment_generator import TreatmentGenerator
from src.privacy.differential_privacy import DifferentialPrivacy

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ All imports successful!")

## 1. Generate Synthetic Patient Data

Let's start by generating synthetic patient demographic data.

In [ ]:
# Generate 1000 synthetic patients
generator = PatientGenerator(num_patients=1000, seed=42)
patients_df = generator.generate_patients()

print(f"Generated {len(patients_df)} patients")
patients_df.head()

### Data Quality Checks

In [ ]:
# Check for missing values
print("Missing values:")
print(patients_df.isnull().sum())

# Check data types
print("\nData types:")
print(patients_df.dtypes)

### Visualize Demographics

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Age distribution
axes[0, 0].hist(patients_df['age'], bins=30, edgecolor='black')
axes[0, 0].set_title('Age Distribution')
axes[0, 0].set_xlabel('Age')
axes[0, 0].set_ylabel('Frequency')

# Gender distribution
patients_df['gender'].value_counts().plot(kind='bar', ax=axes[0, 1])
axes[0, 1].set_title('Gender Distribution')
axes[0, 1].set_xlabel('Gender')
axes[0, 1].set_ylabel('Count')

# Insurance distribution
patients_df['insurance'].value_counts().plot(kind='bar', ax=axes[0, 2])
axes[0, 2].set_title('Insurance Distribution')
axes[0, 2].set_xlabel('Insurance Type')
axes[0, 2].set_ylabel('Count')
axes[0, 2].tick_params(axis='x', rotation=45)

# Income distribution
axes[1, 0].hist(patients_df['income'], bins=30, edgecolor='black')
axes[1, 0].set_title('Income Distribution')
axes[1, 0].set_xlabel('Income ($)')
axes[1, 0].set_ylabel('Frequency')

# Condition prevalence
conditions = ['diabetes', 'hypertension', 'heart_disease']
prevalence = patients_df[conditions].sum()
prevalence.plot(kind='bar', ax=axes[1, 1])
axes[1, 1].set_title('Condition Prevalence')
axes[1, 1].set_xlabel('Condition')
axes[1, 1].set_ylabel('Count')
axes[1, 1].tick_params(axis='x', rotation=45)

# Age vs Income
axes[1, 2].scatter(patients_df['age'], patients_df['income'], alpha=0.3)
axes[1, 2].set_title('Age vs Income')
axes[1, 2].set_xlabel('Age')
axes[1, 2].set_ylabel('Income ($)')

plt.tight_layout()
plt.show()

## 2. Generate Vital Signs

Now let's generate realistic vital signs for our patients.

In [ ]:
vitals_gen = VitalsGenerator()
vitals_df = vitals_gen.generate_vitals(patients_df)

print(f"Generated vitals for {len(vitals_df)} patients")
vitals_df.head()

### Visualize Vital Signs

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Blood Pressure
axes[0, 0].scatter(vitals_df['blood_pressure_systolic'], 
                   vitals_df['blood_pressure_diastolic'], alpha=0.5)
axes[0, 0].set_title('Blood Pressure')
axes[0, 0].set_xlabel('Systolic (mmHg)')
axes[0, 0].set_ylabel('Diastolic (mmHg)')

# Blood Glucose
axes[0, 1].hist(vitals_df['blood_glucose'], bins=30, edgecolor='black')
axes[0, 1].set_title('Blood Glucose Distribution')
axes[0, 1].set_xlabel('Blood Glucose (mg/dL)')
axes[0, 1].set_ylabel('Frequency')

# Heart Rate
axes[1, 0].hist(vitals_df['heart_rate'], bins=30, edgecolor='black')
axes[1, 0].set_title('Heart Rate Distribution')
axes[1, 0].set_xlabel('Heart Rate (bpm)')
axes[1, 0].set_ylabel('Frequency')

# Oxygen Saturation
axes[1, 1].hist(vitals_df['oxygen_saturation'], bins=30, edgecolor='black')
axes[1, 1].set_title('Oxygen Saturation Distribution')
axes[1, 1].set_xlabel('O2 Saturation (%)')
axes[1, 1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## 3. Apply Differential Privacy

Let's apply differential privacy to protect sensitive patient information.

In [ ]:
# Initialize differential privacy with epsilon=1.0
dp = DifferentialPrivacy(epsilon=1.0, delta=1e-5)

# Apply privacy to age and income columns
private_patients = dp.privatize_dataframe(
    patients_df,
    numeric_columns=['age', 'income'],
    categorical_columns=['insurance']
)

print("Applied differential privacy to patient data")
private_patients.head()

### Compare Original vs Private Data

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Age comparison
axes[0].hist(patients_df['age'], bins=30, alpha=0.5, label='Original', edgecolor='black')
axes[0].hist(private_patients['age'], bins=30, alpha=0.5, label='Private', edgecolor='black')
axes[0].set_title('Age Distribution: Original vs Private')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Income comparison
axes[1].hist(patients_df['income'], bins=30, alpha=0.5, label='Original', edgecolor='black')
axes[1].hist(private_patients['income'], bins=30, alpha=0.5, label='Private', edgecolor='black')
axes[1].set_title('Income Distribution: Original vs Private')
axes[1].set_xlabel('Income ($)')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

### Compute Private Statistics

In [ ]:
# Compute private statistics
private_mean_age = dp.private_mean(patients_df['age'].values)
private_var_age = dp.private_variance(patients_df['age'].values)
private_count = dp.private_count(patients_df['age'].values)

print("Private Statistics (ε=1.0, δ=1e-5):")
print(f"  Mean Age: {private_mean_age:.2f} (True: {patients_df['age'].mean():.2f})")
print(f"  Variance: {private_var_age:.2f} (True: {patients_df['age'].var():.2f})")
print(f"  Count: {private_count:.0f} (True: {len(patients_df)})")

## 4. Simulate Disease Progression

Let's simulate disease progression for a patient with diabetes.

In [ ]:
# Find a patient with diabetes
diabetic_patient = patients_df[patients_df['diabetes'] == 1].iloc[0]

print(f"Selected patient: {diabetic_patient['patient_id']}")
print(f"Age: {diabetic_patient['age']}, Gender: {diabetic_patient['gender']}")
print(f"Conditions: Diabetes={diabetic_patient['diabetes']}, Hypertension={diabetic_patient['hypertension']}")

# Simulate 12 visits over 1 year
model = DiseaseProgressionModel()
progression_df = model.simulate_progression(
    diabetic_patient,
    num_visits=12,
    time_interval_days=30
)

print(f"\nSimulated {len(progression_df)} visits")
progression_df.head()

### Visualize Disease Progression

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Blood Glucose over time
axes[0, 0].plot(progression_df['visit_number'], progression_df['blood_glucose'], marker='o')
axes[0, 0].set_title('Blood Glucose Progression')
axes[0, 0].set_xlabel('Visit Number')
axes[0, 0].set_ylabel('Blood Glucose (mg/dL)')
axes[0, 0].grid(True)

# Blood Pressure over time
axes[0, 1].plot(progression_df['visit_number'], progression_df['blood_pressure_systolic'], 
                marker='o', label='Systolic')
axes[0, 1].plot(progression_df['visit_number'], progression_df['blood_pressure_diastolic'], 
                marker='o', label='Diastolic')
axes[0, 1].set_title('Blood Pressure Progression')
axes[0, 1].set_xlabel('Visit Number')
axes[0, 1].set_ylabel('Blood Pressure (mmHg)')
axes[0, 1].legend()
axes[0, 1].grid(True)

# HbA1c over time
axes[1, 0].plot(progression_df['visit_number'], progression_df['hemoglobin_a1c'], marker='o')
axes[1, 0].set_title('HbA1c Progression')
axes[1, 0].set_xlabel('Visit Number')
axes[1, 0].set_ylabel('HbA1c (%)')
axes[1, 0].axhline(y=7.0, color='r', linestyle='--', label='Target < 7%')
axes[1, 0].legend()
axes[1, 0].grid(True)

# Weight over time
axes[1, 1].plot(progression_df['visit_number'], progression_df['weight'], marker='o')
axes[1, 1].set_title('Weight Progression')
axes[1, 1].set_xlabel('Visit Number')
axes[1, 1].set_ylabel('Weight (lbs)')
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

## 5. Generate Treatments

Finally, let's assign treatments based on patient conditions.

In [ ]:
treatment_gen = TreatmentGenerator()
treatments_df = treatment_gen.generate_treatments(patients_df)

print(f"Generated treatments for {len(treatments_df)} patients")
treatments_df.head()

### Treatment Distribution

In [ ]:
# Count most common treatments
all_treatments = []
for treatments in treatments_df['treatments']:
    all_treatments.extend(treatments)

treatment_counts = pd.Series(all_treatments).value_counts().head(10)

plt.figure(figsize=(12, 6))
treatment_counts.plot(kind='bar')
plt.title('Top 10 Most Common Treatments')
plt.xlabel('Treatment')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("\nTreatment statistics:")
print(f"Total treatments prescribed: {len(all_treatments)}")
print(f"Unique treatments: {len(set(all_treatments))}")
print(f"Average treatments per patient: {len(all_treatments) / len(treatments_df):.2f}")

## Summary

In this notebook, we demonstrated:

1. ✓ Generating 1000 synthetic patients with realistic demographics
2. ✓ Creating vital signs that account for patient conditions
3. ✓ Applying differential privacy to protect sensitive data
4. ✓ Simulating disease progression over time
5. ✓ Generating treatment assignments based on conditions

All data generated is synthetic and suitable for research, development, and testing purposes while maintaining HIPAA compliance through differential privacy.